In [ ]:
import re
import os
import unicodedata
from datasets import load_dataset, Dataset
import pyarrow as pa
import pyarrow.parquet as pq

In [ ]:
# =================CONFIGURATION=================
SOURCE_DATASET = "ai4bharat/sangraha"
SOURCE_SUBSET = "verified"
LANG_CODE = "tam"
OUTPUT_DIR = "processed_data/sangraha_tamil_clean"
HF_REPO_ID = "Azri-Muhsin/sangraha-tamil-cleaned"
BATCH_SIZE = 100_000
# ===============================================

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
def clean_text_sillama_style(text):
    """
    Applies SiLlama-style cleaning rules adapted for Tamil:
    1. Unicode Normalization (NFC)
    2. Remove HTML tags if any present
    3. Script Identification (Must be primarily Tamil)
    4. Length filtering
    5. Agressively filtering noise phrases
    """
    if not isinstance(text, str):
        return None

    # 1. Unicode Normalization
    text = unicodedata.normalize('NFC', text)

    # 5. Noise Filtering
    if re.search(r'home screen|amazon card |Click here', text, flags=re.IGNORECASE):
        return None

    # 2. Heuristic Cleaning (Remove HTML & Excessive Whitespace)
    text = re.sub(r'<.*?>', '', text) # Remove HTML tags
    text = re.sub(r'^ - ', '', text)  # Remove leading hyphens
    text = re.sub(r'\s+', ' ', text).strip() # Collapse whitespace


    # 3. Filter Short/Empty Sentences (< 20 chars)
    if len(text) < 20:
        return None

    # 4. Script Filtering
    # Check if a significant portion of the text is actually Tamil.
    # Tamil Unicode Block: U+0B80 to U+0BFF
    tamil_char_count = len(re.findall(r'[\u0B80-\u0BFF]', text))
    total_char_count = len(text.replace(" ", ""))

    if total_char_count == 0:
        return None

    # Keep only if > 50% of characters are Tamil
    if (tamil_char_count / total_char_count) < 0.5:
        return None

    return text

def process_and_upload():
    print(f"Loading {SOURCE_DATASET} (verified/{LANG_CODE}) in streaming mode...")

    # Sangraha structure requires pointing to data_dir for specific language to avoid downloading all 22 langs
    # Note: If 'verified/tam' fails, we fallback to streaming 'verified' and filtering.

    dataset = load_dataset(
            SOURCE_DATASET,
            data_dir=f"verified/{LANG_CODE}",
            split="train",
            streaming=True
    )


    buffer = []
    chunk_counter = 0
    total_processed = 0

    # Schema for Parquet (Simple text column)
    schema = pa.schema([('text', pa.string())])

    print("Starting processing...")

    for i, sample in enumerate(dataset):
        # Extract text (handle different column names if necessary)
        raw_text = sample.get('text', sample.get('content', ''))

        cleaned_text = clean_text_sillama_style(raw_text)

        if cleaned_text:
            buffer.append({'text': cleaned_text})

        # Flush to disk every BATCH_SIZE
        if len(buffer) >= BATCH_SIZE:
            chunk_filename = os.path.join(OUTPUT_DIR, f"part-{chunk_counter}.parquet")

            # Convert buffer to PyArrow Table
            table = pa.Table.from_pylist(buffer, schema=schema)
            pq.write_table(table, chunk_filename)

            print(f"Saved chunk {chunk_counter} ({len(buffer)} rows) to {chunk_filename}")
            buffer = []
            chunk_counter += 1
            total_processed += BATCH_SIZE

    # Flush remaining buffer
    if buffer:
        chunk_filename = os.path.join(OUTPUT_DIR, f"part-{chunk_counter}.parquet")
        table = pa.Table.from_pylist(buffer, schema=schema)
        pq.write_table(table, chunk_filename)
        print(f"Saved final chunk {chunk_counter} ({len(buffer)} rows).")

    print(f"Processing complete! Total estimated rows: {total_processed + len(buffer)}")

    # Upload to Hub
    print(f"Uploading to Hugging Face Hub: {HF_REPO_ID}...")

    # We load the local folder as a dataset to push it easily
    final_dataset = load_dataset("parquet", data_files=f"{OUTPUT_DIR}/*.parquet")
    final_dataset.push_to_hub(HF_REPO_ID, private=False) # Set private=False if you want it public

    print("Upload Complete! ✨")

if __name__ == "__main__":
    process_and_upload()

Loading ai4bharat/sangraha (verified/tam) in streaming mode...


Resolving data files:   0%|          | 0/53 [00:00<?, ?it/s]

Starting processing...
Saved chunk 0 (100000 rows) to processed_data/sangraha_tamil_clean/part-0.parquet
Saved chunk 1 (100000 rows) to processed_data/sangraha_tamil_clean/part-1.parquet
Saved chunk 2 (100000 rows) to processed_data/sangraha_tamil_clean/part-2.parquet
Saved chunk 3 (100000 rows) to processed_data/sangraha_tamil_clean/part-3.parquet
Saved chunk 4 (100000 rows) to processed_data/sangraha_tamil_clean/part-4.parquet
Saved chunk 5 (100000 rows) to processed_data/sangraha_tamil_clean/part-5.parquet
Saved chunk 6 (100000 rows) to processed_data/sangraha_tamil_clean/part-6.parquet
Saved chunk 7 (100000 rows) to processed_data/sangraha_tamil_clean/part-7.parquet
Saved chunk 8 (100000 rows) to processed_data/sangraha_tamil_clean/part-8.parquet
Saved chunk 9 (100000 rows) to processed_data/sangraha_tamil_clean/part-9.parquet
Saved chunk 10 (100000 rows) to processed_data/sangraha_tamil_clean/part-10.parquet
Saved chunk 11 (100000 rows) to processed_data/sangraha_tamil_clean/part-

Resolving data files:   0%|          | 0/78 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/78 [00:00<?, ?it/s]

HfHubHTTPError: (Request ID: Root=1-695b4fb2-6f4c16a03677fcca293a511d;96f4fb2a-27a4-4383-b50d-b402cf8fe607)

403 Forbidden: You don't have the rights to create a dataset under the namespace "Azri-Muhsin".
Cannot access content at: https://huggingface.co/api/repos/create.
Make sure your token has the correct permissions.

In [ ]:
from huggingface_hub import HfApi


MY_WRITE_TOKEN = "secret"

OUTPUT_DIR = "processed_data/sangraha_tamil_clean"
HF_REPO_ID = "Azri-Muhsin/sangraha-tamil-cleaned"

print(f"Force-uploading to {HF_REPO_ID} using explicit token...")

# --- 2. UPLOAD WITH EXPLICIT TOKEN ---
api = HfApi(token=MY_WRITE_TOKEN)

try:
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HF_REPO_ID,
        repo_type="dataset",
        commit_message="Upload processed Tamil chunks via Manual Token"
    )
    print("🚀 Upload Complete! Success.")
except Exception as e:
    print(f"❌ Failed: {e}")
    print("\nTroubleshooting:")
    print("1. Did you manually create the repo on the website first? (Recommended)")
    print("2. Are you SURE the token in the variable above is the Write token?")

Force-uploading to Azri-Muhsin/sangraha-tamil-cleaned using explicit token...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mil_clean/part-10.parquet:   3%|3         | 7.87MB /  234MB            

  ...mil_clean/part-12.parquet:   1%|          | 1.57MB /  233MB            

...

  ...mil_clean/part-11.parquet:   1%|1         | 2.62MB /  235MB            

  ...mil_clean/part-14.parquet:   0%|          | 1.05MB /  233MB            

  ...mil_clean/part-16.parquet:   5%|5         | 12.1MB /  235MB            

  ...mil_clean/part-18.parquet:   2%|1         | 3.67MB /  234MB            

🚀 Upload Complete! Success.


In [ ]:
# Delete the Hugging Face dataset cache
!rm -rf /root/.cache/huggingface/datasets

!rm -rf /root/.cache/huggingface/hub

print("Hugging Face dataset and model cache cleared!")

Hugging Face dataset and model cache cleared!


In [ ]:
import shutil
import os


if os.path.exists("processed_data"):
    shutil.rmtree("processed_data")
    print("Deleted 'processed_data' folder.")

Deleted 'processed_data' folder.


In [ ]:
# Show disk usage of the current directory
!du -h --max-depth=1 /content

# Show overall disk status (Free vs Used)
!df -h /

140K	/content/.config
55M	/content/sample_data
55M	/content
Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   71G   38G  66% /
